# Label Forge - Google Colab Training Notebook

This notebook trains YOLO models on Colab GPU and syncs metrics back to Label Forge.

In [ ]:
# @title Setup Parameters
JOB_ID = "job_id_here"  # @param {type:"string"}
DATASET_URL = "https://backend-public.example.com/datasets/xyz.zip"  # @param {type:"string"}
CALLBACK_URL = "https://backend-public.example.com/training/job_id/colab-callback"  # @param {type:"string"}

print(f"Job ID: {JOB_ID}")
print(f"Dataset URL: {DATASET_URL}")
print(f"Callback URL: {CALLBACK_URL}")

In [ ]:
# @title Validate Public URLs
from urllib.parse import urlparse

def is_local_url(url: str) -> bool:
    parsed = urlparse(url)
    host = parsed.hostname or ""
    return host in {"localhost", "127.0.0.1", "0.0.0.0"}

if is_local_url(DATASET_URL) or is_local_url(CALLBACK_URL):
    raise RuntimeError(
        "DATASET_URL and CALLBACK_URL must be public URLs. Set BACKEND_PUBLIC_URL on the backend (for example, a cloudflared https://xxxxx.trycloudflare.com URL) and reopen the Colab link."
    )

print("✓ Public URLs detected. Training can proceed.")

In [ ]:
# @title Install Dependencies
print("Installing dependencies...")
!pip install -q ultralytics torch torchvision requests gdown pyyaml

In [ ]:
# @title Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# @title Download Dataset
import os
import io
import zipfile
import requests

print("Downloading dataset...")
work_dir = f"/content/labelforge_train_{JOB_ID}"
dataset_dir = os.path.join(work_dir, "dataset")
os.makedirs(dataset_dir, exist_ok=True)

response = requests.get(DATASET_URL, stream=True, timeout=120)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    zip_ref.extractall(dataset_dir)

print(f"✓ Dataset extracted to {dataset_dir}")
print(f"  Files: {os.listdir(dataset_dir)}")

data_yaml = os.path.join(dataset_dir, "data.yaml")
if os.path.exists(data_yaml):
    print("✓ data.yaml found")
    with open(data_yaml, "r") as f:
        print(f"  Content:\n{f.read()}")
else:
    raise RuntimeError("data.yaml not found in dataset ZIP")

In [ ]:
# @title YOLO Training
from ultralytics import YOLO
from datetime import datetime

print("\n" + "=" * 60)
print("STARTING YOLO TRAINING")
print("=" * 60)

architecture = "yolov8s"
image_size = 640
batch_size = 16
epochs = 50
learning_rate = 0.006
patience = 12

runs_dir = os.path.join(work_dir, "runs")
device = 0

print(f"Model: {architecture}")
print(f"Image Size: {image_size}")
print(f"Batch Size: {batch_size}")
print(f"Epochs: {epochs}")
print(f"Learning Rate: {learning_rate}")

model = YOLO(f"{architecture}.pt")
print(f"\nTraining started at {datetime.now().isoformat()}")
train_results = model.train(
    data=data_yaml,
    epochs=epochs,
    imgsz=image_size,
    batch=batch_size,
    lr0=learning_rate,
    patience=patience,
    project=runs_dir,
    name="train",
    exist_ok=True,
    device=device,
    workers=8,
    plots=False,
    verbose=True,
)
print(f"\nTraining completed at {datetime.now().isoformat()}")
print("=" * 60)

In [ ]:
# @title Evaluate Model & Extract Metrics
print("\nEvaluating model on validation set...")

save_dir = getattr(train_results, "save_dir", None) or os.path.join(runs_dir, "train")
best_model_path = os.path.join(str(save_dir), "weights", "best.pt")

if not os.path.exists(best_model_path):
    raise RuntimeError(f"best.pt not found at {best_model_path}")

trained_model = YOLO(best_model_path)
validation = trained_model.val(data=data_yaml, imgsz=image_size, verbose=False)
metrics = getattr(validation, "results_dict", {}) or {}

def get_metric_value(metrics_dict, candidates):
    for key in candidates:
        value = metrics_dict.get(key)
        if value is not None:
            try:
                return float(value)
            except (TypeError, ValueError):
                pass
    return 0.0

map_score = get_metric_value(metrics, ["metrics/mAP50-95(B)", "metrics/mAP50-95"])
precision = get_metric_value(metrics, ["metrics/precision(B)", "metrics/precision"])
recall = get_metric_value(metrics, ["metrics/recall(B)", "metrics/recall"])

print(f"mAP50-95: {map_score:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

training_metrics = {
    "map_score": float(map_score),
    "precision": float(precision),
    "recall": float(recall),
    "epochs": epochs,
}

In [ ]:
# @title Generate Sample Predictions
import glob

print("\nGenerating sample predictions on validation images...")

valid_images_dir = os.path.join(dataset_dir, "valid", "images")
sample_predictions = []

if os.path.isdir(valid_images_dir):
    image_paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
        image_paths.extend(glob.glob(os.path.join(valid_images_dir, ext)))
        if len(image_paths) >= 6:
            break
    image_paths = image_paths[:6]
    print(f"Found {len(image_paths)} validation images")
    if image_paths:
        confidence_threshold = 0.25
        names = getattr(trained_model, "names", {}) or {}
        results = trained_model.predict(image_paths, conf=confidence_threshold, verbose=False)
        for result in results:
            image_name = os.path.basename(getattr(result, "path", "sample.jpg"))
            boxes = getattr(result, "boxes", None)
            if boxes is None:
                continue
            for box in boxes[:1]:
                cls_idx = int(box.cls[0].item())
                xywhn = box.xywhn[0].tolist()
                sample_predictions.append({
                    "image_name": image_name,
                    "class_name": names.get(cls_idx, str(cls_idx)),
                    "confidence": round(float(box.conf[0].item()), 4),
                    "bbox": {
                        "x": round(float(xywhn[0] - xywhn[2] / 2), 4),
                        "y": round(float(xywhn[1] - xywhn[3] / 2), 4),
                        "width": round(float(xywhn[2]), 4),
                        "height": round(float(xywhn[3]), 4),
                    },
                })
        print(f"✓ Generated {len(sample_predictions)} sample predictions")
else:
    print("No validation images found")

training_metrics["sample_predictions"] = sample_predictions[:6]

In [ ]:
# @title Send Results Back to Label Forge
import requests
import json

print("Sending results back to Label Forge...")
payload = {
    "status": "done",
    "metrics": {
        "map_score": training_metrics["map_score"],
        "precision": training_metrics["precision"],
        "recall": training_metrics["recall"],
    },
    "sample_predictions": training_metrics["sample_predictions"],
    "model_url": f"minio://model-artifacts/{JOB_ID}/best.pt",
}

print(json.dumps(payload, indent=2))
response = requests.post(CALLBACK_URL, json=payload, timeout=30)
print(f"Callback status: {response.status_code}")
response.raise_for_status()
print("✓ Results successfully sent to Label Forge")

## Training Complete

Results have been sent back to Label Forge. Check the training dashboard for metrics and predictions.